In [1]:
import json
import sys
import dLux
import collections
from observation import Dither

structure = json.load(open('structure.json', 'r'))

dLux: Jax is running in 32-bit, to enable 64-bit visit: https://jax.readthedocs.io/en/latest/notebooks/Common_Gotchas_in_JAX.html#double-64bit-precision


In [2]:
def get_class(modules_str):
    modules = modules_str.split('.')

    # Get the main module
    if modules[0] == '__main__':
        main = sys.modules['__main__']
    elif modules[0] == 'dict':
        return dict
    else:
      main = getattr(sys.modules['__main__'], modules[0])

    # Get the final class type
    sub_modules = modules[1:]
    for module in sub_modules:
        module = getattr(main, module)
    return module

def get_class_type(class_str):
    classes = class_str.split("'")
    if len(classes) == 1:
        return classes
    else:
        class_type = classes[1]
    return get_class(class_type)

def construct_classes(structure):
    class_type = structure['type']
    class_parameters = structure['parameters']

    # Construct an empty class
    class_type = get_class_type(class_type)
    class_types = {"Main": class_type}

    # Populate the class with parameters
    for key, value in class_parameters.items():
        
        # Node
        if isinstance(value, dict): 
            structure_dict = value
            class_type = structure_dict['type']
            class_parameters = structure_dict['parameters']
            class_types[key] = construct_classes(structure_dict)
        
        # Leaf
        else: 
            class_types[key] = value
    return class_types

In [3]:
class_structure = construct_classes(structure)

In [4]:
class_structure

{'Main': dLux.core.Instrument,
 'optics': {'Main': dLux.core.Optics,
  'layers': {'Main': collections.OrderedDict,
   'CreateWavefront': {'Main': dLux.optics.CreateWavefront,
    'name': "<class 'str'>",
    'npixels': "<class 'int'>",
    'diameter': "<class 'jaxlib.xla_extension.Array'>",
    'wavefront_type': "<class 'str'>"},
   'CircularAperture': {'Main': dLux.apertures.StaticAperture,
    'name': "<class 'str'>",
    'aperture': "<class 'jaxlib.xla_extension.Array'>"},
   'NormaliseWavefront': {'Main': dLux.optics.NormaliseWavefront,
    'name': "<class 'str'>"},
   'AngularMFT': {'Main': dLux.propagators.AngularMFT,
    'name': "<class 'str'>",
    'inverse': "<class 'bool'>",
    'pixel_scale_out': "<class 'jaxlib.xla_extension.Array'>",
    'npixels_out': "<class 'int'>",
    'shift': "<class 'jaxlib.xla_extension.Array'>",
    'pixel_shift': "<class 'bool'>"}}},
 'detector': "<class 'NoneType'>",
 'scene': {'Main': dLux.core.Scene,
  'sources': {'Main': dict,
   'PointSource

In [5]:
structure

{'type': "<class 'dLux.core.Instrument'>",
 'parameters': {'optics': {'type': "<class 'dLux.core.Optics'>",
   'parameters': {'layers': {'type': "<class 'collections.OrderedDict'>",
     'parameters': {'CreateWavefront': {'type': "<class 'dLux.optics.CreateWavefront'>",
       'parameters': {'name': "<class 'str'>",
        'npixels': "<class 'int'>",
        'diameter': "<class 'jaxlib.xla_extension.Array'>",
        'wavefront_type': "<class 'str'>"}},
      'CircularAperture': {'type': "<class 'dLux.apertures.StaticAperture'>",
       'parameters': {'name': "<class 'str'>",
        'aperture': "<class 'jaxlib.xla_extension.Array'>"}},
      'NormaliseWavefront': {'type': "<class 'dLux.optics.NormaliseWavefront'>",
       'parameters': {'name': "<class 'str'>"}},
      'AngularMFT': {'type': "<class 'dLux.propagators.AngularMFT'>",
       'parameters': {'name': "<class 'str'>",
        'inverse': "<class 'bool'>",
        'pixel_scale_out': "<class 'jaxlib.xla_extension.Array'>",
   

In [67]:
import zodiax
import jax.numpy as np

def construct_class(instance):
    if hasattr(instance, '_construct'):
        return instance._construct()
    else:
        return instance

def get_class(modules_str):

    if modules_str == 'dict':
        return {}
    elif modules_str == 'str':
        return ''
    elif modules_str == 'int':
        return 0
    elif modules_str == 'NoneType':
        return None
    elif modules_str == 'bool':
        return False
    elif modules_str == 'collections.OrderedDict':
        return collections.OrderedDict()
    elif modules_str == 'jaxlib.xla_extension.Array':
        return np.array([])
    
    # dLux
    elif modules_str.startswith('dLux'):
        module = getattr(sys.modules['__main__'], 'dLux')
        for sub_module in modules_str.split('.')[1:]:
            module = getattr(module, sub_module)
        return module._construct()
    
    # Some other module - Search for it
    else:
        modules = modules_str.split('.')
        module = getattr(sys.modules['__main__'], modules[1])
        for sub_module in modules_str.split('.')[2:]:
            module = getattr(module, sub_module)
        return construct_class(module)

def get_class_type(class_str):
    classes = class_str.split("'")
    if len(classes) == 1:
        return classes
    else:
        class_type = classes[1]
    return get_class(class_type)

def construct_classes(structure):
    class_type = structure['type']
    class_parameters = structure['parameters']

    # Construct an empty class
    class_type = get_class_type(class_type)
    outer_class = class_type

    # Populate the class with parameters
    for key, value in class_parameters.items():
        
        if isinstance(value, dict): # Node
            structure_dict = value
            class_type = structure_dict['type']
            class_parameters = structure_dict['parameters']
            inner_class = construct_classes(structure_dict)
            
        else: # Leaf
            inner_class = get_class_type(value)
        
        if isinstance(outer_class, zodiax.Base):
            if inner_class is None: # None case - Zodiax bug
                outer_class = outer_class.set([key], [inner_class])
            else:
                outer_class = outer_class.set(key, inner_class)
        elif isinstance(outer_class, dict):
            outer_class[key] = inner_class
    return outer_class

In [74]:
import equinox as eqx
tel_struc = construct_classes(structure)
tel = eqx.tree_deserialise_leaves("tel.eqx", tel_struc)

RuntimeError: Deserialised leaf has changed shape from (0,) in `like` to () on disk.

In [75]:
tel

Instrument(
  optics=Optics(
    layers={
      'CreateWavefront':
      CreateWavefront(name='', npixels=0, diameter=f32[0], wavefront_type=''),
      'CircularAperture':
      StaticAperture(name='', aperture=f32[0]),
      'NormaliseWavefront':
      NormaliseWavefront(name=''),
      'AngularMFT':
      AngularMFT(
        name='',
        inverse=False,
        npixels_out=0,
        pixel_scale_out=f32[0],
        shift=f32[0],
        pixel_shift=False
      )
    }
  ),
  scene=Scene(
    sources={
      'PointSource':
      PointSource(
        position=f32[0],
        flux=f32[0],
        spectrum=ArraySpectrum(name='', wavelengths=f32[0], weights=f32[0]),
        name=''
      )
    }
  ),
  detector=None,
  filter=None,
  observation=Dither(name='', dithers=f32[0])
)